<a href="https://colab.research.google.com/github/vasilevsavelij135-spec/python-ai-template/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`Экономические показатели стран.csv`](https://github.com/vasilevsavelij135-spec/python-ai-template/blob/main/data/Экономические%20показатели%20стран.csv) — экономические показатели стран: ВВП, население, координаты

**Что мы делаем:**

1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [5]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-template"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-template
    !git clone -q https://github.com/vasilevsavelij135-spec/python-ai-template.git  # ← ИЗМЕНЕНО: новый URL репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-template


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [8]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_genre = pd.read_csv("data/Экономические показатели стран.csv")

print("✅ Загружено строк в df_genre:", len(df_genre))

✅ Загружено строк в df_genre: 222


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технический столбец** `country` с URL (ссылкой на объект Wikidata), который мешает простому анализу.

- Столбец `country` с URL — **удаляем**, так как для базового анализа он не нужен.
- Столбец `countryLabel` содержит читаемое название страны — **переименуем в `country`** для удобства.

В этом шаге мы:
- удалим столбец с URL Wikidata (`country`);
- переименуем `countryLabel → country`;
- приведём числовые столбцы (`gdp`, `population`) к типу `float`/`int`, если потребуется.

При приведении к числам мы используем:
- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` или `astype(float)` — переводит столбец к нужному числовому типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец с URL можно сохранить, если нужно будет быстро перейти к оригинальной записи в Викиданных.

In [11]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для экономических данных

# Initialize df_economy with the data currently in df_genre
df_economy = df_genre.copy()

# Работаем с df_economy: ВВП, население, координаты
if "countryLabel" in df_economy.columns:                      # ← ИЗМЕНЕНО: проверка нужного столбца
    # Удаляем технический столбец с URL Wikidata
    if "country" in df_economy.columns:                       # ← ДОБАВЛЕНО: безопасное удаление URL
        df_economy = df_economy.drop(columns=["country"])

    # Переименовываем Label-столбец в читаемое название
    df_economy = df_economy.rename(columns={
        "countryLabel": "country",                             # ← ИЗМЕНЕНО: новое имя столбца
    })

    # Приводим числовые столбцы к правильному типу
    df_economy["gdp"] = pd.to_numeric(
        df_economy["gdp"], errors="coerce"
    ).fillna(0).astype(float)                                  # ← ИЗМЕНЕНО: обработка gdp

    df_economy["population"] = pd.to_numeric(
        df_economy["population"], errors="coerce"
    ).fillna(0).astype(int)                                    # ← ИЗМЕНЕНО: обработка population

    print("✅ df_economy очищен и переименован")
else:
    print("⏭️ df_economy уже очищен, пропускаем")

# Быстрая проверка результата
print("\n📋 Столбцы после очистки:", df_economy.columns.tolist())
print("✅ Данные готовы к анализу")

✅ df_economy очищен и переименован

📋 Столбцы после очистки: ['country', 'gdp', 'population', 'coordinates']
✅ Данные готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с экономическими показателями:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по числовым полям (`gdp`, `population`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код.

**Что содержит наш DataFrame `df_economy`:**
- `country` — название страны (строка);
- `gdp` — ВВП страны (число, млрд $);
- `population` — население (целое число);
- `coordinates` — географические координаты (строка формата WKT).

> 💡 **Совет:** функция `describe()` в pandas автоматически покажет статистику только по числовым столбцам, что удобно для быстрого анализа экономических показателей.

In [12]:
# 🔍 Шаг 3. Обзор данных

def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 📋 Обзор основного DataFrame
show_info(df_economy, "Экономические показатели стран (df_economy)")  # ← ИЗМЕНЕНО: один DataFrame

# 📈 Базовая статистика по числовым столбцам (gdp, population)
print("\n📈 Базовая статистика:")
print(df_economy[["gdp", "population"]].describe())  # ← ДОБАВЛЕНО: статистика по вашим числовым полям

# 🗺️ Бонус: быстрая проверка координат (если планируем строить карту)
if "coordinates" in df_economy.columns:
    print("\n🌍 Примеры координат:")
    print(df_economy["coordinates"].head(3))  # ← ДОБАВЛЕНО: просмотр формата координат


📊 Экономические показатели стран (df_economy)
Размер: (222, 4)
Столбцы: country, gdp, population, coordinates

Первые строки:
          country       gdp  population         coordinates
0       Австралия  54348.23    27614411  Point(133.0 -25.0)
1  Новая Зеландия  41666.64     5307800  Point(174.0 -41.2)
2         Венгрия  37128.00     9603634    Point(19.0 47.0)
3           Чехия  31707.00    10909500    Point(15.0 50.0)
4         Испания  29993.06    48592909    Point(-3.5 40.2)

📈 Базовая статистика:
                gdp    population
count    222.000000  2.220000e+02
mean    1112.012432  3.607280e+07
std     6263.141814  1.436151e+08
min        0.000000  0.000000e+00
25%        0.000000  6.334065e+05
50%        0.000000  6.130056e+06
75%        0.000000  2.372832e+07
max    54348.230000  1.477520e+09

🌍 Примеры координат:
0    Point(133.0 -25.0)
1    Point(174.0 -41.2)
2      Point(19.0 47.0)
Name: coordinates, dtype: object


## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных стран** представлено в данных;
- **какие страны встречаются** в датасете (проверка на дубликаты);
- **базовую статистику по ВВП и населению**: мин, макс, среднее;
- **есть ли пропущенные значения** в ключевых столбцах;
- **формат координат**: все ли записи корректны.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_economy["country"].value_counts().head()` даёт **список стран по частоте встречаемости** (помогает найти дубликаты названий).

> 💡 **Совет:** если одна страна встречается несколько раз с разными значениями ВВП — возможно, данные за разные годы. Если с одинаковыми — возможно, дубликаты, которые нужно удалить.

Метод `isna().sum()` покажет количество пропущенных значений в каждом столбце — это важно для экономического анализа, где пропуски могут исказить выводы.

In [13]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка экономических данных")

# 📊 Основной датасет: экономические показатели стран
print("\n📊 Датасет: Экономические показатели стран (df_economy)")
print("Уникальных стран:", df_economy["country"].nunique())  # ← ИЗМЕНЕНО: проверка стран
print("Всего записей:", len(df_economy))

# 🔎 Проверка на дубликаты названий стран
print("\n🔍 Частота встречаемости стран (проверка на дубликаты):")
print(df_economy["country"].value_counts().head(10))  # ← ИЗМЕНЕНО: анализ повторов

# 📈 Статистика по числовым показателям
print("\n📈 Статистика по ВВП (gdp):")
print(df_economy["gdp"].describe())  # ← ИЗМЕНЕНО: статистика по ВВП

print("\n📈 Статистика по населению (population):")
print(df_economy["population"].describe())  # ← ИЗМЕНЕНО: статистика по населению

# ⚠️ Проверка на пропущенные значения
print("\n⚠️ Пропущенные значения в столбцах:")
print(df_economy.isna().sum())  # ← ДОБАВЛЕНО: важная проверка качества данных

# 🗺️ Валидация формата координат (если столбец есть)
if "coordinates" in df_economy.columns:
    print("\n🌍 Проверка формата координат:")
    # Проверяем, все ли записи содержат "Point("
    valid_coords = df_economy["coordinates"].str.contains("Point\\(", na=False).sum()
    print(f"Записей с корректным форматом координат: {valid_coords} из {len(df_economy)}")
    print("Примеры координат:")
    print(df_economy["coordinates"].head(3))  # ← ДОБАВЛЕНО: визуальная проверка

# 🎯 Бонус: Топ-10 стран по ВВП (быстрый инсайт)
print("\n🏆 Топ-10 стран по ВВП:")
print(df_economy.nlargest(10, "gdp")[["country", "gdp"]])  # ← ДОБАВЛЕНО: полезный вывод

🔍 Быстрая проверка экономических данных

📊 Датасет: Экономические показатели стран (df_economy)
Уникальных стран: 221
Всего записей: 222

🔍 Частота встречаемости стран (проверка на дубликаты):
country
Британский Северный Борнео    2
Новая Зеландия                1
Венгрия                       1
Чехия                         1
Австралия                     1
Эстония                       1
Барбадос                      1
Ботсвана                      1
Папуа — Новая Гвинея          1
Боливия                       1
Name: count, dtype: int64

📈 Статистика по ВВП (gdp):
count      222.000000
mean      1112.012432
std       6263.141814
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      54348.230000
Name: gdp, dtype: float64

📈 Статистика по населению (population):
count    2.220000e+02
mean     3.607280e+07
std      1.436151e+08
min      0.000000e+00
25%      6.334065e+05
50%      6.130056e+06
75%      2.372832e+07
max      1.477520e+09
Name: 

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨